# Generador de Base de Datos Sintética — Riesgo Académico

Este notebook construye un dataset **artificial pero estadísticamente coherente**
que simula los registros que se obtendrían de un LMS tipo Moodle, siguiendo la
tabla de variables (features) diseñada previamente para el modelo predictivo
de riesgo académico.

**Objetivo:** generar `dataset.csv` en la carpeta `MobileLearning` con datos
listos para explorar, entrenar y validar algoritmos de clasificación (riesgo
de deserción / reprobación).

**Diseño general:**
- Cada fila representa un *snapshot* (corte temporal) de un estudiante en un
  curso específico, no un registro único por estudiante. Esto permite que el
  dataset tenga estructura temporal, necesaria para features como la
  tendencia de calificaciones o los días de inactividad acumulados.
- Las variables no se generan de forma independiente: se apoyan en variables
  *latentes* (habilidad académica y nivel de compromiso/engagement) que no
  aparecen en el dataset final, pero que correlacionan de forma realista el
  rendimiento con el comportamiento — igual que ocurriría con estudiantes
  reales.
- La variable objetivo (`riesgo_academico_confirmado`) se calcula con una
  función logística sobre los factores de riesgo, no de forma arbitraria, de
  modo que el dataset sea útil para probar algoritmos de clasificación
  supervisada de verdad.


In [1]:
# --- 1. Librerías ---
# numpy y pandas: generación numérica y ensamblado tabular.
# scipy.stats: cálculo de pendientes (tendencia) y distribuciones beta/gamma.
# pathlib: manejo de rutas de forma portable (Windows/Linux/Mac).
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path
from datetime import datetime, timedelta

# Semilla global para que el dataset sea reproducible: si vuelves a correr
# el notebook, obtienes exactamente los mismos datos.
SEED = 42
rng = np.random.default_rng(SEED)


## 2. Parámetros de la simulación

Aquí se define el "tamaño" del mundo simulado: cuántos estudiantes, cuántos
cursos existen y cuántos cortes temporales (snapshots) se registran por cada
matrícula. Cambiar estos números es suficiente para generar datasets más
grandes o más pequeños sin tocar el resto del notebook.

In [2]:
# --- 2. Parámetros generales ---
N_ESTUDIANTES = 400                 # tamaño de la población simulada
CURSOS = [f"CUR-{i:03d}" for i in range(1, 9)]   # 8 cursos disponibles en el ciclo
N_CORTES = 4                         # snapshots por matrícula (ej. cada 3 semanas)
FECHA_INICIO_CICLO = datetime(2026, 3, 16)       # inicio del semestre académico
DIAS_ENTRE_CORTES = 21               # separación entre cada snapshot (3 semanas)

# Cada estudiante se matricula en un número variable de cursos (carga académica).
# Se usa una distribución discreta con más peso en 4-5 cursos, que es lo típico.
CARGA_POSIBLE = np.array([3, 4, 5, 6, 7])
PROB_CARGA = np.array([0.10, 0.30, 0.35, 0.18, 0.07])


## 3. Variables latentes por estudiante

No forman parte del dataset final, pero son el "motor" que hace que las
variables observables (calificaciones, inactividad, participación, etc.)
estén correlacionadas entre sí de forma realista:

- **`habilidad_latente`**: capacidad académica de base del estudiante (0-1).
- **`compromiso_latente`**: disposición a mantenerse activo, entregar a
  tiempo y participar (0-1). Está parcialmente correlacionado con la
  habilidad (un estudiante hábil tiende a ser algo más disciplinado), pero
  se le añade ruido propio porque en la realidad hay estudiantes hábiles y
  desconectados, y estudiantes muy comprometidos pero con más dificultades.

In [3]:
# --- 3. Variables latentes (no observables, uso interno) ---
habilidad_latente = rng.beta(a=5, b=3, size=N_ESTUDIANTES)  # sesgada hacia valores medios-altos

# El compromiso se construye como una mezcla de la habilidad (correlación
# parcial) más un componente aleatorio independiente, para que ambos rasgos
# no queden perfectamente correlacionados.
ruido_compromiso = rng.beta(a=4, b=4, size=N_ESTUDIANTES)
compromiso_latente = np.clip(0.5 * habilidad_latente + 0.5 * ruido_compromiso, 0, 1)

df_estudiantes = pd.DataFrame({
    "id_estudiante": [f"EST-{i:04d}" for i in range(1, N_ESTUDIANTES + 1)],
    "habilidad_latente": habilidad_latente,
    "compromiso_latente": compromiso_latente,
})
df_estudiantes.head()


,id_estudiante,habilidad_latente,compromiso_latente
0,EST-0001,0.567062,0.375514
1,EST-0002,0.652155,0.614242
2,EST-0003,0.711192,0.583709
3,EST-0004,0.634336,0.716347
4,EST-0005,0.741994,0.565671


## 4. Matrícula: qué cursos toma cada estudiante

Se asigna a cada estudiante una carga académica (número de cursos) siguiendo
la distribución definida arriba, y luego se eligen esos cursos al azar entre
los disponibles. Esto genera el campo `carga_academica_actual` directamente
desde el proceso de matrícula, en lugar de simularlo aparte.

In [4]:
# --- 4. Simulación de matrícula ---
matriculas = []
for _, fila in df_estudiantes.iterrows():
    n_cursos = rng.choice(CARGA_POSIBLE, p=PROB_CARGA)
    cursos_asignados = rng.choice(CURSOS, size=n_cursos, replace=False)
    for curso in cursos_asignados:
        matriculas.append({
            "id_estudiante": fila["id_estudiante"],
            "id_curso": curso,
            "carga_academica_actual": n_cursos,
            "habilidad_latente": fila["habilidad_latente"],
            "compromiso_latente": fila["compromiso_latente"],
        })

df_matricula = pd.DataFrame(matriculas)
print(f"Total de matrículas (estudiante-curso): {len(df_matricula)}")
df_matricula.head()


Total de matrículas (estudiante-curso): 1877


,id_estudiante,id_curso,carga_academica_actual,habilidad_latente,compromiso_latente
0,EST-0001,CUR-007,5,0.567062,0.375514
1,EST-0001,CUR-008,5,0.567062,0.375514
2,EST-0001,CUR-003,5,0.567062,0.375514
3,EST-0001,CUR-001,5,0.567062,0.375514
4,EST-0001,CUR-002,5,0.567062,0.375514


## 5. Expansión temporal: snapshots por matrícula

Cada matrícula se repite `N_CORTES` veces, una por cada fecha de corte del
ciclo. Esto es lo que le da al dataset su estructura de panel (mismo
estudiante-curso observado en el tiempo), indispensable para calcular
tendencias y evitar fuga de información (*data leakage*) hacia el pasado.

In [5]:
# --- 5. Expansión a snapshots temporales ---
fechas_corte = [FECHA_INICIO_CICLO + timedelta(days=DIAS_ENTRE_CORTES * k)
                for k in range(1, N_CORTES + 1)]

filas_expandidas = []
for _, fila in df_matricula.iterrows():
    for corte_idx, fecha in enumerate(fechas_corte):
        registro = fila.to_dict()
        registro["corte_idx"] = corte_idx        # 0, 1, 2, 3 -> orden temporal
        registro["fecha_corte"] = fecha.strftime("%Y-%m-%d")
        filas_expandidas.append(registro)

df = pd.DataFrame(filas_expandidas)
print(f"Total de filas (snapshots): {len(df)}")
df.head()


Total de filas (snapshots): 7508


,id_estudiante,id_curso,carga_academica_actual,habilidad_latente,compromiso_latente,corte_idx,fecha_corte
0,EST-0001,CUR-007,5,0.567062,0.375514,0,2026-04-06
1,EST-0001,CUR-007,5,0.567062,0.375514,1,2026-04-27
2,EST-0001,CUR-007,5,0.567062,0.375514,2,2026-05-18
3,EST-0001,CUR-007,5,0.567062,0.375514,3,2026-06-08
4,EST-0001,CUR-008,5,0.567062,0.375514,0,2026-04-06


## 6. Variables de rendimiento académico

- `calificacion_historica_promedio`: parte de la habilidad latente (escala
  0-20, como en el sistema peruano) y se le añade una leve caída progresiva
  a lo largo de los cortes para los estudiantes de menor compromiso —
  simulando que el desgaste se acumula con el tiempo.
- `tasa_acierto_primer_intento`: correlacionada con la habilidad, pero con su
  propio ruido, porque mide precisión al responder, no solo conocimiento.

In [6]:
# --- 6. Rendimiento académico ---
# Calificación base a partir de la habilidad (escala 0-20).
calif_base = df["habilidad_latente"] * 20

# Efecto de desgaste: a mayor índice de corte y menor compromiso, mayor caída.
efecto_desgaste = (df["corte_idx"] + 1) * (1 - df["compromiso_latente"]) * 0.8

ruido_calif = rng.normal(0, 1.3, size=len(df))
df["calificacion_historica_promedio"] = np.clip(
    calif_base - efecto_desgaste + ruido_calif, 0, 20
).round(2)

# Tasa de acierto en el primer intento: proporción (0-1), distribución beta
# ligada a la habilidad para que no sea una copia lineal de la calificación.
alpha = 2 + df["habilidad_latente"] * 6
beta_param = 2 + (1 - df["habilidad_latente"]) * 4
df["tasa_acierto_primer_intento"] = rng.beta(alpha, beta_param).round(3)


## 7. Tendencia de calificaciones (regresión por grupo)

En lugar de asignar la tendencia al azar, se calcula de verdad: para cada
combinación estudiante-curso se ajusta una **regresión lineal simple**
(`scipy.stats.linregress`) entre el índice de corte (tiempo) y la
calificación observada. La pendiente resultante es la tendencia real de esos
4 puntos, exactamente como se calcularía sobre datos reales de Moodle.

In [7]:
# --- 7. Cálculo de tendencia real por regresión lineal ---
def calcular_pendiente(grupo):
    x = grupo["corte_idx"].values
    y = grupo["calificacion_historica_promedio"].values
    if len(x) < 2:
        return 0.0
    pendiente, _, _, _, _ = stats.linregress(x, y)
    return round(pendiente, 3)

tendencias = (
    df.groupby(["id_estudiante", "id_curso"])
    .apply(calcular_pendiente)
    .rename("tendencia_calificaciones")
)

df = df.merge(tendencias, on=["id_estudiante", "id_curso"], how="left")
df[["id_estudiante", "id_curso", "corte_idx",
    "calificacion_historica_promedio", "tendencia_calificaciones"]].head(8)


C:\Users\renzo\AppData\Local\Temp\ipykernel_16620\3993612536.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calcular_pendiente)


,id_estudiante,id_curso,corte_idx,calificacion_historica_promedio,tendencia_calificaciones
0,EST-0001,CUR-007,0,11.32,-1.109
1,EST-0001,CUR-007,1,9.94,-1.109
2,EST-0001,CUR-007,2,10.97,-1.109
3,EST-0001,CUR-007,3,7.28,-1.109
4,EST-0001,CUR-008,0,10.14,-0.730
5,EST-0001,CUR-008,1,13.00,-0.730
6,EST-0001,CUR-008,2,9.03,-0.730
7,EST-0001,CUR-008,3,9.03,-0.730


## 8. Variables de comportamiento y compromiso

Estas variables dependen sobre todo del `compromiso_latente`, no de la
habilidad, porque reflejan hábitos de estudio y no capacidad intelectual:

- `dias_inactividad_consecutivos`: distribución de Poisson invertida por
  compromiso — a menor compromiso, más días sin conectarse.
- `tasa_entregas_tardias`, `nivel_interaccion_social`,
  `tasa_visualizacion_materiales`: proporciones (0-1) generadas con
  distribución beta, cada una con su propia relación con el compromiso.

In [9]:
# --- 8. Comportamiento en la plataforma ---
# Días de inactividad: media alta cuando el compromiso es bajo (Poisson).
lambda_inactividad = 1 + (1 - df["compromiso_latente"]) * 12
df["dias_inactividad_consecutivos"] = rng.poisson(lambda_inactividad)

# Entregas tardías: proporción que sube cuando baja el compromiso.
a_tardio = 1 + (1 - df["compromiso_latente"]) * 5
b_tardio = 1 + df["compromiso_latente"] * 5
df["tasa_entregas_tardias"] = rng.beta(a_tardio, b_tardio).round(3)

# Interacción social en foros: sube con el compromiso.
a_social = 1 + df["compromiso_latente"] * 5
b_social = 1 + (1 - df["compromiso_latente"]) * 5
df["nivel_interaccion_social"] = rng.beta(a_social, b_social).round(3)

# Visualización de materiales obligatorios: también ligada al compromiso,
# con algo más de ruido porque no siempre depende solo de la actitud del
# estudiante (a veces el material tarda en publicarse, por ejemplo).
a_vis = 1 + df["compromiso_latente"] * 4
b_vis = 1 + (1 - df["compromiso_latente"]) * 3
df["tasa_visualizacion_materiales"] = rng.beta(a_vis, b_vis).round(3)


## 9. Variables adicionales de mayor costo de extracción

Estas dos variables (duración del intento e índice de fricción cognitiva)
fueron marcadas como valiosas pero costosas de calcular en la tabla original.
Se incluyen igualmente para que el dataset esté completo, generadas con
distribuciones distintas (exponencial y gamma) para diversificar los
algoritmos usados en el notebook.

In [10]:
# --- 9. Duración de intento e índice de fricción cognitiva ---
# Duración del intento (minutos): distribución exponencial, más corta cuando
# la habilidad es alta (responde con seguridad) y más larga en los extremos
# de baja habilidad (dificultad real) — se modela con una media variable.
media_duracion = 15 - df["habilidad_latente"] * 6
df["duracion_intento_minutos"] = np.round(rng.exponential(media_duracion), 1)

# Índice de fricción cognitiva: número promedio de cambios de respuesta antes
# de enviar. Se modela con distribución gamma, más alta cuando la habilidad
# es intermedia (mucha inseguridad) y más baja en los extremos.
inseguridad = 1 - np.abs(df["habilidad_latente"] - 0.5) * 2  # pico en habilidad media
df["indice_friccion_cognitiva"] = np.round(
    rng.gamma(shape=1.5 + inseguridad * 2, scale=0.8), 2
)


## 10. Variable objetivo: riesgo académico confirmado

El riesgo no se asigna al azar: se calcula con una **función logística**
sobre una combinación ponderada de los factores de riesgo (estandarizados),
tal como lo haría un modelo de regresión logística real. Después se muestrea
un valor binario con `np.random.binomial` usando esa probabilidad, para que
el dataset conserve incertidumbre realista (dos estudiantes con el mismo
perfil no siempre terminan en el mismo desenlace).

In [11]:
# --- 10. Cálculo del riesgo académico (target) ---
def estandarizar(serie):
    return (serie - serie.mean()) / serie.std()

# Factores que AUMENTAN el riesgo (signo positivo) y factores que lo
# DISMINUYEN (signo negativo), ponderados según su relevancia teórica.
z_riesgo = (
      1.4 * estandarizar(df["dias_inactividad_consecutivos"])
    + 1.1 * estandarizar(df["tasa_entregas_tardias"])
    + 0.6 * estandarizar(df["indice_friccion_cognitiva"])
    - 1.3 * estandarizar(df["calificacion_historica_promedio"])
    - 0.9 * estandarizar(df["tendencia_calificaciones"])
    - 0.7 * estandarizar(df["tasa_acierto_primer_intento"])
    - 0.5 * estandarizar(df["nivel_interaccion_social"])
    - 0.4 * estandarizar(df["tasa_visualizacion_materiales"])
    - 2.5                                        # intercepto: calibra la prevalencia base (~20%)
    + rng.normal(0, 0.6, size=len(df))          # ruido irreducible
)

# Función logística (sigmoide) que transforma el puntaje z en probabilidad.
probabilidad_riesgo = 1 / (1 + np.exp(-z_riesgo))

# El intercepto añadido arriba calibra la prevalencia de riesgo alrededor de
# 25-30%, un rango plausible para tasas de deserción/reprobación en educación
# superior (sin él, al estar todos los factores estandarizados, la
# probabilidad promedio quedaría en ~50%, lo cual no es realista). Puedes
# ajustar el valor del intercepto si necesitas una prevalencia distinta.
df["riesgo_academico_confirmado"] = rng.binomial(1, probabilidad_riesgo)

print("Proporción de riesgo confirmado:")
print(df["riesgo_academico_confirmado"].value_counts(normalize=True).round(3))


Proporción de riesgo confirmado:
riesgo_academico_confirmado
0    0.728
1    0.272
Name: proportion, dtype: float64


## 11. Ensamblado final y validaciones

Se seleccionan y ordenan las columnas finales según la tabla de diseño, y se
corren validaciones básicas (sin nulos, rangos coherentes) antes de exportar.
Esto evita descubrir errores de generación recién en la etapa de modelado.

In [12]:
# --- 11. Selección final de columnas ---
columnas_finales = [
    "id_estudiante", "id_curso", "fecha_corte", "carga_academica_actual",
    "calificacion_historica_promedio", "tendencia_calificaciones",
    "tasa_acierto_primer_intento", "duracion_intento_minutos",
    "indice_friccion_cognitiva", "dias_inactividad_consecutivos",
    "tasa_entregas_tardias", "nivel_interaccion_social",
    "tasa_visualizacion_materiales", "riesgo_academico_confirmado",
]
dataset_final = df[columnas_finales].copy()

# --- Validaciones básicas ---
assert dataset_final.isnull().sum().sum() == 0, "Hay valores nulos inesperados"
assert dataset_final["calificacion_historica_promedio"].between(0, 20).all()
assert dataset_final["riesgo_academico_confirmado"].isin([0, 1]).all()
for col in ["tasa_acierto_primer_intento", "tasa_entregas_tardias",
            "nivel_interaccion_social", "tasa_visualizacion_materiales"]:
    assert dataset_final[col].between(0, 1).all(), f"{col} fuera de rango [0,1]"

print(f"Filas: {len(dataset_final)} | Columnas: {len(dataset_final.columns)}")
dataset_final.describe().round(2)


Filas: 7508 | Columnas: 14


,carga_academica_actual,calificacion_historica_promedio,tendencia_calificaciones,tasa_acierto_primer_intento,duracion_intento_minutos,indice_friccion_cognitiva,dias_inactividad_consecutivos,tasa_entregas_tardias,nivel_interaccion_social,tasa_visualizacion_materiales,riesgo_academico_confirmado
count,7508.00,7508.00,7508.00,7508.00,7508.00,7508.00,7508.00,7508.00,7508.00,7508.00,7508.00
mean,4.90,11.49,-0.36,0.62,11.23,2.23,6.27,0.45,0.54,0.58,0.27
std,0.99,3.78,0.60,0.17,11.27,1.37,2.88,0.19,0.19,0.21,0.44
min,3.00,0.00,-2.16,0.05,0.00,0.02,0.00,0.02,0.02,0.01,0.00
25%,4.00,9.00,-0.77,0.50,3.10,1.21,4.00,0.30,0.40,0.43,0.00
50%,5.00,11.76,-0.36,0.63,7.90,1.96,6.00,0.45,0.55,0.60,0.00
75%,6.00,14.32,0.04,0.75,15.50,2.98,8.00,0.59,0.69,0.74,1.00
max,7.00,20.00,1.81,0.99,108.60,10.45,22.00,0.96,0.99,1.00,1.00


In [13]:
# Se elimina la columna objetivo del dataset exportado.
# Se hace en este punto (después de las validaciones) y no antes, para no
# afectar los asserts que verifican el rango de riesgo_academico_confirmado.
dataset_final = dataset_final.drop(columns=["riesgo_academico_confirmado"])

print(f"Columnas finales tras eliminar el target: {list(dataset_final.columns)}")
dataset_final.head()

Columnas finales tras eliminar el target: ['id_estudiante', 'id_curso', 'fecha_corte', 'carga_academica_actual', 'calificacion_historica_promedio', 'tendencia_calificaciones', 'tasa_acierto_primer_intento', 'duracion_intento_minutos', 'indice_friccion_cognitiva', 'dias_inactividad_consecutivos', 'tasa_entregas_tardias', 'nivel_interaccion_social', 'tasa_visualizacion_materiales']


,id_estudiante,id_curso,fecha_corte,carga_academica_actual,calificacion_historica_promedio,tendencia_calificaciones,tasa_acierto_primer_intento,duracion_intento_minutos,indice_friccion_cognitiva,dias_inactividad_consecutivos,tasa_entregas_tardias,nivel_interaccion_social,tasa_visualizacion_materiales
0,EST-0001,CUR-007,2026-04-06,5,11.32,-1.109,0.584,15.7,5.04,8,0.525,0.472,0.698
1,EST-0001,CUR-007,2026-04-27,5,9.94,-1.109,0.256,8.1,1.29,8,0.772,0.426,0.341
2,EST-0001,CUR-007,2026-05-18,5,10.97,-1.109,0.578,11.1,2.66,9,0.835,0.697,0.282
3,EST-0001,CUR-007,2026-06-08,5,7.28,-1.109,0.639,2.9,1.64,8,0.681,0.484,0.539
4,EST-0001,CUR-008,2026-04-06,5,10.14,-0.730,0.702,5.9,2.22,5,0.558,0.461,0.521


## 12. Exportación a `dataset.csv`

El archivo se guarda dentro de la misma carpeta `MobileLearning` donde vive
este notebook, usando una ruta relativa para que funcione sin cambios sin
importar en qué máquina se ejecute.

In [14]:
# --- 12. Exportación final ---
carpeta_salida = Path.cwd()          # carpeta MobileLearning (donde corre el notebook)
ruta_csv = carpeta_salida / "dataset.csv"

dataset_final.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
print(f"Dataset exportado exitosamente en: {ruta_csv.resolve()}")
dataset_final.head(10)


Dataset exportado exitosamente en: C:\Users\renzo\Documents\MobileLearning\dataset.csv


,id_estudiante,id_curso,fecha_corte,carga_academica_actual,calificacion_historica_promedio,tendencia_calificaciones,tasa_acierto_primer_intento,duracion_intento_minutos,indice_friccion_cognitiva,dias_inactividad_consecutivos,tasa_entregas_tardias,nivel_interaccion_social,tasa_visualizacion_materiales
0,EST-0001,CUR-007,2026-04-06,5,11.32,-1.109,0.584,15.7,5.04,8,0.525,0.472,0.698
1,EST-0001,CUR-007,2026-04-27,5,9.94,-1.109,0.256,8.1,1.29,8,0.772,0.426,0.341
2,EST-0001,CUR-007,2026-05-18,5,10.97,-1.109,0.578,11.1,2.66,9,0.835,0.697,0.282
3,EST-0001,CUR-007,2026-06-08,5,7.28,-1.109,0.639,2.9,1.64,8,0.681,0.484,0.539
4,EST-0001,CUR-008,2026-04-06,5,10.14,-0.730,0.702,5.9,2.22,5,0.558,0.461,0.521
5,EST-0001,CUR-008,2026-04-27,5,13.00,-0.730,0.646,34.1,1.88,8,0.667,0.312,0.281
6,EST-0001,CUR-008,2026-05-18,5,9.03,-0.730,0.669,10.6,1.18,7,0.804,0.668,0.726
7,EST-0001,CUR-008,2026-06-08,5,9.03,-0.730,0.480,0.2,3.30,7,0.783,0.270,0.538
8,EST-0001,CUR-003,2026-04-06,5,9.69,-0.366,0.487,1.4,2.29,8,0.804,0.676,0.439
9,EST-0001,CUR-003,2026-04-27,5,10.78,-0.366,0.624,1.4,2.93,11,0.698,0.468,0.309
